## Get Dataset

In [ ]:
import pandas as pd

df = pd.read_parquet("hf://datasets/RevolutionCrossroads/si_us_revolutionary_era_collections/si_revwar.parquet")

sil_df = df[df.apply(lambda row: len(row["mediaURLs"]) == 1 and "silhouette" in str(row["indexed_object_types"]).lower(), axis=1)].copy()

sil_df["id"] = sil_df.apply(lambda row: f"000000{row.name}"[-5:], axis=1)
sil_df["uuid"] = sil_df["guid"].apply(lambda x: x.split("/")[-1].replace("-",""))
sil_df["img_url"] = sil_df["mediaURLs"].apply(lambda x: x[0])

sil_exp_df = sil_df[["id", "uuid", "img_url", "thumbnail"]]

sil_exp_df.to_csv("./image/rev-sils/sils_info.csv", index=False)

## Download Images

In [ ]:
import pandas as pd
import requests

def save_img(image_url, fpath):
  img_data = requests.get(image_url).content
  with open(fpath, "wb") as handler:
    handler.write(img_data)

df = pd.read_csv("./image/rev-sils/sils_info.csv", dtype={"id": str})

for idx,row in df.iterrows():
  fpath = f"./00_orig/{row['id']}.jpg"
  save_img(row["img_url"], fpath)

## Crop images and save raw contours

In [ ]:
import cv2
import json
import numpy as np
import pandas as pd

from PIL import Image as PImage

In [ ]:
def contour_is_valid(c, h, w, m=1):
  for p in c:
    x, y = p[0]
    if x < m or x > w - m - 1 or y < m or y > h - m - 1:
      return False
  return cv2.contourArea(c) < 0.80 * h * w

center_r = 10

df = pd.read_csv("./image/rev-sils/sils_info.csv", dtype={"id": str})

contour_data_cropped_raw = []

for idx,row in list(df.iterrows()):
  img = PImage.open(f"./00_orig/{row['id']}.jpg")
  iw,ih = img.size

  img_np = np.array(img.resize((iw//4, ih//4)))
  nph,npw,_ = img_np.shape

  center = img_np[nph//2-center_r:nph//2+center_r+1, npw//2-center_r:npw//2+center_r+1]
  center_avg = int(center.mean())

  ret, img_t_np = cv2.threshold(cv2.cvtColor(img_np, cv2.COLOR_BGR2GRAY), center_avg+32, 255, cv2.THRESH_BINARY)
  contours, hierarchy = cv2.findContours(image=img_t_np, mode=cv2.RETR_TREE, method=cv2.CHAIN_APPROX_NONE)

  if contours:
    filtered_contours = [c for c in contours if contour_is_valid(c, nph, npw)]
    largest_contour = max(filtered_contours, key=cv2.contourArea)

    cv2.drawContours(img_np, [largest_contour], -1, (0, 255, 0), 1)
    bx, by, bw, bh = cv2.boundingRect(largest_contour)
    cv2.rectangle(img_np, (bx, by), (bx + bw, by + bh), (0, 0, 255), 2)

    cropped = img.crop((4*bx, 4*by, 4*(bx+bw), 4*(by+bh)))
    cw, ch = cropped.size

    if cw > 255 and ch > 255:
      cropped.save(f"./image/rev-sils/01_cropped/{row['id']}.jpg")
      contour_data_cropped_raw.append({
        "id": row["id"],
        "contour": [[int(px-bx)*4, int(py-by)*4] for px,py in largest_contour.reshape(-1, 2).tolist()],
        "size": [cw, ch]
      })
    else:
      cropped.save(f"./image/rev-sils/01_cropped/fail/{row['id']}.jpg")

In [ ]:
with open("./sils_cropped_raw.json", "w") as ofp:
  json.dump(contour_data_cropped_raw, ofp)

## Export images with consistent height

In [ ]:
import json
import numpy as np

from PIL import Image as PImage

In [ ]:
with open("./sils_cropped_raw.json", "r") as ifp:
  contour_data_cropped_raw = json.load(ifp)

min_w, min_h = np.array([x["size"] for x in contour_data_cropped_raw]).min(axis=0)

for img in contour_data_cropped_raw:
  iw,ih = img["size"]
  mid = img["id"]
  img = PImage.open(f"./image/rev-sils/01_cropped/{mid}.jpg")
  iw,ih = img.size
  nw = int(iw/ih * min_h)
  img.resize((nw, min_h)).save(f"./image/rev-sils/h452/{mid}.jpg")

## Load contours for processing

In [ ]:
import json
import numpy as np
import pandas as pd

from sklearn.cluster import KMeans

In [ ]:
with open("./sils_cropped_raw.json", "r") as ifp:
  contour_data_cropped_raw = json.load(ifp)

min_contour_len = min([len(c) for c in [x["contour"] for x in contour_data_cropped_raw]])
ids = [x["id"] for x in contour_data_cropped_raw]

print(min_contour_len)

In [ ]:
def sort_by_angle(points):
  cx,cy = points.mean(axis=0)
  return np.array(sorted(points, key=lambda A: 100*np.arctan2(A[1]-cy, A[0]-cx) + ((A[1]-cy)**2 + (A[0]-cx)**2)**0.5))

def center_points_1d(points):
  avg = (points.max() + points.min()) / 2
  return points - avg

def center_points_2d(points, flatten=False):
  x_points_centered = center_points_1d(points[:, 0])
  y_points_centered = center_points_1d(points[:, 1])
  if flatten:
    return np.stack((x_points_centered, y_points_centered), axis=1).reshape(-1)
  else:
    return np.stack((x_points_centered, y_points_centered), axis=1)

contour_data = []
for img in contour_data_cropped_raw:
  iw,ih = img["size"]
  contour = np.array(img["contour"]) / max(iw,ih)

  kmeans = KMeans(n_clusters=min_contour_len, random_state=1010).fit(contour)
  contour_data.append(center_points_2d(sort_by_angle(kmeans.cluster_centers_), flatten=True))

contour_data_np = np.array(contour_data)
contour_data_np.shape

In [ ]:
col_names = np.array([(f"x{i}", f"y{i}") for i in range(contour_data_np.shape[1]//2)]).reshape(-1).tolist()

sil_con_df = pd.DataFrame(contour_data_np, columns=col_names).astype(float).round(6)
sil_con_df.insert(loc=0, column="id", value=ids)
sil_con_df

sil_con_df.to_csv("./csv/rev_sils_centered.csv", index=False)

## Read centered `DataFrame`

In [ ]:
import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from PIL import Image as PImage, ImageDraw as PImageDraw

sil_con_df = pd.read_csv("./csv/rev_sils_centered.csv", dtype={"id": str})

In [ ]:
ridx = 300
rid = sil_con_df.iloc[ridx]["id"]
rxys = sil_con_df.iloc[ridx].values[1:].reshape(-1,2)

img = PImage.open(f"./image/rev-sils/h452/{rid}.jpg")
iw,ih = img.size
max_dim = max(iw, ih)
draw = PImageDraw.Draw(img)

for x,y in rxys:
  px = x * max_dim + iw//2
  py = y * max_dim + ih//2
  r = 2
  draw.ellipse((px-r, py-r, px+r, py+r), fill=(255,0,0))

img

In [ ]:
rxys = sil_con_df.iloc[:, 1:].mean().values.reshape(-1,2)

img = PImage.fromarray(255*np.ones(shape=(256,256,3), dtype=np.uint8))
iw,ih = img.size
max_dim = max(iw, ih)
draw = PImageDraw.Draw(img)

for x,y in rxys:
  px = x * max_dim + iw//2
  py = y * max_dim + ih//2
  r = 2
  draw.ellipse((px-r, py-r, px+r, py+r), fill=(255,0,0))

img

In [ ]:
sil_con_vals = sil_con_df.drop(columns=["id"]).values

nclusters = 8
kmeans = KMeans(n_clusters=nclusters, random_state=1010).fit(sil_con_vals)

for cidx in range(nclusters):
  aimg = PImage.fromarray(255*np.ones((452,320,3), dtype=np.uint8))
  aiw,aih = aimg.size
  max_dim = max(aiw, aih)
  draw = PImageDraw.Draw(aimg)

  cur_x = 0
  crows = sil_con_df.iloc[kmeans.labels_ == cidx]
  dists = np.linalg.norm(sil_con_vals[kmeans.labels_ == cidx] - kmeans.cluster_centers_[cidx], axis=1)
  top_16 = crows.iloc[np.argsort(dists)][:16]

  cavg = top_16.iloc[:, 1:].mean().values.reshape(-1,2)

  for x,y in cavg:
    px = x * max_dim + aiw//2
    py = y * max_dim + aih//2
    r = 2
    draw.ellipse((px-r, py-r, px+r, py+r), fill=(0,0,0))

  cimg = np.zeros((452,16*452,3), dtype=np.uint8)
  cimg[:, cur_x:cur_x+aiw] = np.array(aimg)

  cur_x = aiw
  for mid in top_16["id"][:16]:
    img = PImage.open(f"./image/rev-sils/h452/{mid}.jpg")
    iw,ih = img.size
    cimg[:, cur_x:cur_x+iw] = np.array(img)
    cur_x += iw

  display(PImage.fromarray(cimg).crop((0,0, cur_x, 452)))